In [1]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
DATA_DIR = Path("../data")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
BATCH_SIZE = 64

d:\myproject\myproject\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
resume_path = DATA_DIR / "processed" / "clean_resume.csv"
job_path = DATA_DIR / "processed" / "clean_job.csv"

if not resume_path.exists():
    resume_path = DATA_DIR / "clean_resume.csv"

if not job_path.exists():
    job_path = DATA_DIR / "clean_job.csv"

resume_df = pd.read_csv(resume_path)
job_df = pd.read_csv(job_path)

In [3]:
print("Resume shape:", resume_df.shape)
print("Job shape:", job_df.shape)

Resume shape: (2481, 5)
Job shape: (180370, 15)


In [4]:
required_resume_cols = ["resume_id", "category", "clean_resume"]
required_job_cols = ["job_id", "title", "full_job"]
for col in required_resume_cols:
    if col not in resume_df.columns:
        raise ValueError(f"Missing required resume column: {col}")
for col in required_job_cols:
    if col not in job_df.columns:
        raise ValueError(f"Missing required job column: {col}")
resume_df["clean_resume"] = resume_df["clean_resume"].fillna("").astype(str)
resume_df["category"] = resume_df["category"].fillna("unknown").astype(str)
job_df["full_job"] = job_df["full_job"].fillna("").astype(str)
job_df["title"] = job_df["title"].fillna("unknown").astype(str)
resume_df.head(2), job_df[["job_id", "title", "full_job"]].head(2)

(   resume_id category                                             resume  \
 0   16852973       HR           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
 1   22323967       HR           HR SPECIALIST, US HR OPERATIONS      ...   
 
                                         clean_resume  resume_word_count  
 0  hr administrator marketing associate hr admini...                658  
 1  hr specialist us hr operations summary versati...                707  ,
    job_id                         title  \
 0       0  Digital Marketing Specialist   
 1       1                 Web Developer   
 
                                             full_job  
 0  digital marketing specialist digital marketing...  
 1  web developer web developer frontend web devel...  )

In [5]:
def normalize_tech_terms(text: str) -> str:
    text = str(text)
    replacements = {
        "c++": " cpp ",
        "c#": " csharp ",
        ".net": " dotnet ",
        "asp.net": " aspdotnet ",
        "node.js": " nodejs ",
        "react.js": " reactjs ",
        "next.js": " nextjs ",
        "vue.js": " vuejs ",
        "express.js": " expressjs ",
        "machine-learning": " machine learning ",
        "deep-learning": " deep learning ",
    }

    text = text.lower()
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def clean_text_light(text: str) -> str:
    text = normalize_tech_terms(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\b\d{10,}\b", " ", text)
    text = re.sub(r"[^a-z0-9+#\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [6]:
resume_texts = resume_df["clean_resume"].tolist()
job_texts = job_df["full_job"].tolist()

print("Num resumes:", len(resume_texts))
print("Num jobs:", len(job_texts))

Num resumes: 2481
Num jobs: 180370


In [7]:
model = SentenceTransformer(EMBED_MODEL_NAME)
print("Loaded model:", EMBED_MODEL_NAME)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3532.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2


In [8]:
job_embeddings = model.encode(
    job_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
job_embeddings = job_embeddings.astype(np.float32)

resume_embeddings = model.encode(
    resume_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
resume_embeddings = resume_embeddings.astype(np.float32)

Batches: 100%|██████████| 39/39 [05:47<00:00,  8.91s/it]


In [28]:
print("Job embeddings shape:", job_embeddings.shape)
print("Resume embeddings shape:", resume_embeddings.shape)

Job embeddings shape: (180370, 384)
Resume embeddings shape: (2481, 384)


In [29]:
np.save(MODEL_DIR / "job_embeddings.npy", job_embeddings)
np.save(MODEL_DIR / "resume_embeddings.npy", resume_embeddings)
job_df[["job_id", "title", "full_job"]].to_csv(MODEL_DIR / "embedding_job_index.csv", index=False)
resume_df[["resume_id", "category", "clean_resume"]].to_csv(MODEL_DIR / "embedding_resume_index.csv", index=False)
config = {
    "model_name": EMBED_MODEL_NAME,
    "batch_size": BATCH_SIZE,
    "embedding_dim": int(job_embeddings.shape[1]),
    "job_embeddings_file": "job_embeddings.npy",
    "resume_embeddings_file": "resume_embeddings.npy",
    "job_index_file": "embedding_job_index.csv",
    "resume_index_file": "embedding_resume_index.csv",
}

with open(MODEL_DIR / "embedding_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
print("Artifacts saved in:", MODEL_DIR)

Artifacts saved in: ..\models


In [30]:
def safe_top_k_indices(scores, top_k=5):
    scores = np.asarray(scores, dtype=np.float32)

    if scores.size == 0:
        return np.array([], dtype=int)

    top_k = min(top_k, scores.size)
    idx = np.argpartition(scores, -top_k)[-top_k:]
    idx = idx[np.argsort(scores[idx])[::-1]]
    return idx


def semantic_score_to_percent(score: float) -> float:
    score = max(0.0, min(1.0, float(score)))
    return round(score * 100, 2)

def _safe_job_id(value):
    try:
        return int(value)
    except Exception:
        return value

def _safe_resume_id(value):
    try:
        return int(value)
    except Exception:
        return value

def encode_query_text(text: str) -> np.ndarray:
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text cannot be empty")

    clean_text = clean_text_light(text)

    if not clean_text:
        raise ValueError("Input text became empty after cleaning")

    emb = model.encode(
        [clean_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return emb[0].astype(np.float32)

In [31]:
def safe_top_k_indices(scores, top_k=5):
    scores = np.asarray(scores, dtype=np.float32)

    if scores.size == 0:
        return np.array([], dtype=int)

    top_k = min(top_k, scores.size)
    idx = np.argpartition(scores, -top_k)[-top_k:]
    idx = idx[np.argsort(scores[idx])[::-1]]
    return idx

def semantic_score_to_percent(score: float) -> float:
    score = max(0.0, min(1.0, float(score)))
    return round(score * 100, 2)

def _safe_job_id(value):
    try:
        return int(value)
    except Exception:
        return value

def _safe_resume_id(value):
    try:
        return int(value)
    except Exception:
        return value

def encode_query_text(text: str) -> np.ndarray:
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text cannot be empty")

    clean_text = clean_text_light(text)

    if not clean_text:
        raise ValueError("Input text became empty after cleaning")

    emb = model.encode(
        [clean_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return emb[0].astype(np.float32)

In [32]:
def _format_job_results(sim_scores, top_k=5, unique_titles=True):
    raw_k = min(len(sim_scores), max(top_k * 5, top_k))
    candidate_idx = safe_top_k_indices(sim_scores, raw_k)

    results = []
    seen_titles = set()

    for idx in candidate_idx:
        row = job_df.iloc[idx]
        title_key = str(row["title"]).strip().lower()

        if unique_titles and title_key in seen_titles:
            continue
        seen_titles.add(title_key)

        results.append({
            "job_id": _safe_job_id(row["job_id"]),
            "job_title": row["title"],
            "semantic_score": semantic_score_to_percent(sim_scores[idx]),
        })

        if len(results) == top_k:
            break

    return pd.DataFrame(results)


def _format_resume_results(sim_scores, top_k=5):
    raw_k = min(len(sim_scores), max(top_k * 5, top_k))
    candidate_idx = safe_top_k_indices(sim_scores, raw_k)

    results = []

    for idx in candidate_idx:
        row = resume_df.iloc[idx]

        preview = row["clean_resume"][:180]
        if len(row["clean_resume"]) > 180:
            preview += "..."

        results.append({
            "resume_id": _safe_resume_id(row["resume_id"]),
            "category": row["category"],
            "semantic_score": semantic_score_to_percent(sim_scores[idx]),
            "resume_preview": preview,
        })

        if len(results) == top_k:
            break

    return pd.DataFrame(results)

In [33]:
def get_top_jobs_semantic(resume_text, top_k=5, unique_titles=True):
    query_emb = encode_query_text(resume_text)
    sim_scores = job_embeddings @ query_emb
    return _format_job_results(sim_scores, top_k=top_k, unique_titles=unique_titles)


def get_top_resumes_semantic(job_text, top_k=5):
    query_emb = encode_query_text(job_text)
    sim_scores = resume_embeddings @ query_emb
    return _format_resume_results(sim_scores, top_k=top_k)


def match_resume_to_job_text_semantic(resume_text, job_text):
    resume_emb = encode_query_text(resume_text)
    job_emb = encode_query_text(job_text)

    score = float(np.dot(resume_emb, job_emb))

    return {
        "semantic_score": semantic_score_to_percent(score)
    }

In [34]:
def top_jobs_for_existing_resume_semantic(resume_id, top_k=5, unique_titles=True):
    matches = np.where(resume_df["resume_id"].astype(str).values == str(resume_id))[0]

    if len(matches) == 0:
        raise ValueError(f"resume_id not found: {resume_id}")

    resume_idx = int(matches[0])
    sim_scores = job_embeddings @ resume_embeddings[resume_idx]
    return _format_job_results(sim_scores, top_k=top_k, unique_titles=unique_titles)


def top_resumes_for_existing_job_semantic(job_id, top_k=5):
    matches = np.where(job_df["job_id"].astype(str).values == str(job_id))[0]

    if len(matches) == 0:
        raise ValueError(f"job_id not found: {job_id}")

    job_idx = int(matches[0])
    sim_scores = resume_embeddings @ job_embeddings[job_idx]
    return _format_resume_results(sim_scores, top_k=top_k)

In [35]:
def load_phase3_artifacts(model_dir=MODEL_DIR):
    with open(model_dir / "embedding_config.json", "r", encoding="utf-8") as f:
        config = json.load(f)

    model = SentenceTransformer(config["model_name"])
    job_embeddings = np.load(model_dir / config["job_embeddings_file"], mmap_mode="r")
    resume_embeddings = np.load(model_dir / config["resume_embeddings_file"], mmap_mode="r")
    job_index = pd.read_csv(model_dir / config["job_index_file"])
    resume_index = pd.read_csv(model_dir / config["resume_index_file"])

    return model, job_embeddings, resume_embeddings, job_index, resume_index, config

In [36]:
print("Top jobs for first resume:")
display(get_top_jobs_semantic(resume_df.iloc[0]["clean_resume"], top_k=5))

print("Top resumes for first job:")
display(get_top_resumes_semantic(job_df.iloc[0]["full_job"], top_k=5))

print("Direct pair score example:")
print(match_resume_to_job_text_semantic(
    resume_df.iloc[0]["clean_resume"],
    job_df.iloc[0]["full_job"],
))

Top jobs for first resume:


,job_id,job_title,semantic_score
0,112050,HR Manager,62.40
1,156317,Human Resources Manager,61.75


Top resumes for first job:


,resume_id,category,semantic_score,resume_preview
0,15479281,APPAREL,67.36,pr event manager summary experienced creative ...
1,16536141,DIGITAL-MEDIA,66.30,interim senior digital marketing strategy mana...
2,14128006,PUBLIC-RELATIONS,65.90,about creative communications professional goo...
3,17132168,DIGITAL-MEDIA,64.54,social engage sales summary my current role re...
4,94492380,DIGITAL-MEDIA,64.16,director of social media marketing executive p...


Direct pair score example:
{'semantic_score': 32.57}
